# ETS Auction Step-by-Step Debug

This notebook prints a full simulation trace for each year and participant.

Default setup keeps 16 participants for 12 years and uses heuristic decisions for all of them.

In [4]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# ==============================
# Top-level debug configuration
# ==============================
SEED = 42
N_YEARS = 12
N_BOTS = 15
N_LEARNING_AGENTS = 1

# Output controls
PRINT_PRESET_SUMMARY = True
PRINT_YEAR_HEADER = True
PRINT_MARKET_DETAILS = True
PRINT_AUCTION_STATS = True
PRINT_LIQUIDATION_DETAILS = True
PRINT_AGENT_DETAILS = True
PRINT_FINAL_SUMMARY = True
PRINT_WARNING_COUNTERS = True
STORE_TRACE_TABLE = True
DISPLAY_YEAR_TABLES = True
DISPLAY_TRACE_TABLE = True

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "configs").exists() and (parent / "src").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.environment.ets_environment import ETSEnvironment
from src.agents import heuristic_policy

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
TECH_NAMES = ["coal", "gas", "onshore_wind", "offshore_wind", "solar"]

In [5]:
def load_config(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def repeat_to_length(items, n):
    if n <= 0:
        return []
    if not items:
        raise ValueError("Cannot repeat an empty template list.")
    return [copy.deepcopy(items[i % len(items)]) for i in range(n)]


def configure_simulation(cfg: dict, n_years: int, n_learning_agents: int, n_bots: int) -> dict:
    cfg = copy.deepcopy(cfg)

    cfg.setdefault("simulation", {})["n_years"] = int(n_years)

    companies = cfg.setdefault("companies", {})
    budget = cfg.setdefault("budget", {})
    bots = cfg.setdefault("bots", {})

    base_learn_mixes = companies.get("initial_mix", [])
    base_learn_weights = companies.get("reward_weights", [])

    if n_learning_agents > 0:
        companies["initial_mix"] = repeat_to_length(base_learn_mixes, n_learning_agents)
        companies["reward_weights"] = repeat_to_length(base_learn_weights, n_learning_agents)
    else:
        companies["initial_mix"] = []
        companies["reward_weights"] = []

    bot_mix_templates = companies.get("bot_initial_mix", []) or base_learn_mixes
    bot_weight_templates = companies.get("bot_reward_weights", []) or base_learn_weights

    companies["bot_initial_mix"] = repeat_to_length(bot_mix_templates, n_bots)
    companies["bot_reward_weights"] = repeat_to_length(bot_weight_templates, n_bots)

    annual_templates = budget.get("annual_budgets", []) or [800.0]
    capex_templates = budget.get("capex_throughputs", []) or [130.0]

    budget["annual_budgets"] = repeat_to_length(annual_templates, n_learning_agents)
    budget["capex_throughputs"] = repeat_to_length(capex_templates, n_learning_agents)
    budget["bot_annual_budgets"] = repeat_to_length(annual_templates, n_bots)
    budget["bot_capex_throughputs"] = repeat_to_length(capex_templates, n_bots)

    if "urgency_denominators" in bots and bots["urgency_denominators"]:
        bots["urgency_denominators"] = repeat_to_length(bots["urgency_denominators"], n_bots)

    companies["n_agents"] = int(n_learning_agents)
    companies["n_bot_agents"] = int(n_bots)

    return cfg


raw_config = load_config(CONFIG_PATH)
config = configure_simulation(
    raw_config,
    n_years=N_YEARS,
    n_learning_agents=N_LEARNING_AGENTS,
    n_bots=N_BOTS,
)

env = ETSEnvironment(config, seed=SEED)
obs_phase1, _ = env.reset(seed=SEED)

print(f"Using config: {CONFIG_PATH}")
print(f"Participants: learning={env.n_agents}, bots={env.n_bots}, total={env.n_total}")
print(f"Years: {config['simulation']['n_years']}")

Market calibration: 16 participants, emissions=49.6 Mt, cap=50.6 Mt
[ETSEnvironment] 1 learning + 15 bot = 16 total agents | cancel_under_subscribed=False
[ETSEnvironment] Initial bank seed example (episode-start allowance holdings, unit: MtCO2 allowances): A1=1.38, B1=1.26, B2=1.42, B3=1.03, B4=0.93, B5=0.39, B6=0.45, B7=0.20, B8=0.16, B9=1.04, B10=1.53, B11=0.81, B12=1.01, B13=0.34, B14=0.29, B15=0.19
[ETSEnvironment] Context: this is each active agent's starting bank before year-1 actions/compliance; total TNAC seed=12.43 MtCO2.
Using config: c:\Users\danie\Documents\Python_Scripts\Master Thesis\Thesis-Energy-Auction\ets_marl_happo_auction\configs\default.yaml
Participants: learning=1, bots=15, total=16
Years: 12


In [6]:
def participant_name(env: ETSEnvironment, idx: int) -> str:
    if idx < env.n_agents:
        return f"A{idx + 1}"
    return f"B{idx - env.n_agents + 1}"


def fmt_mix(company) -> str:
    return ", ".join([f"{TECH_NAMES[t]}={company.mix[t] * 100:.1f}%" for t in range(5)])


def as_float_list(values, digits=4):
    return [round(float(v), digits) for v in values]


def build_learning_auction_actions(env: ETSEnvironment, config: dict) -> np.ndarray:
    actions = np.zeros((env.n_agents, 10), dtype=np.float32)

    for i in range(env.n_agents):
        company = env.companies[i]
        price_ma3 = env._compute_price_ma3()
        current_year = env.current_year
        cap_t = env.cap_schedule.get_cap(current_year)

        base_penalty_rate = float(config["penalty"]["rate"])
        inflation_rate = float(env._inflation_rate(current_year))
        this_year_auction_volume = env.cap_schedule.preview_auction_volume(
            current_year,
            clearing_price=env.last_clearing_price,
            price_max=float(config["auction"]["price_max"]),
            penalty_rate=base_penalty_rate,
            inflation_rate=inflation_rate,
            price_ma3=price_ma3,
        )
        # Mirror ETSEnvironment._get_obs_phase1: include both rollover channels.
        unsold_pending = float(getattr(env.cap_schedule, "_unsold_rollover_pending", 0.0))
        defaulted_pending = float(env._defaulted_volume_pending)
        this_year_auction_volume += unsold_pending + defaulted_pending
        max_rollover_mult = float(getattr(env.cap_schedule, "max_rollover_multiplier", 1.5))
        this_year_auction_volume = min(this_year_auction_volume, cap_t * max_rollover_mult)

        action6 = heuristic_policy.auction_action(
            company,
            price_ma3,
            current_year,
            config["simulation"]["n_years"],
            config,
            bank=float(env.holdings[i]),
            reserve_price=env._compute_dynamic_reserve(),
            auction_volume=float(this_year_auction_volume),
            cap_t=float(cap_t),
            suspension_remaining=int(env._suspension_remaining[i]),
            suspension_length=int(config["auction"].get("suspension_length", 2)),
            collateral_load_last=float(env._last_collateral_load[i]),
        )

        p_mid = float(action6[0])
        q_total = float(action6[1])
        q_third = q_total / 3.0
        p1 = float(np.clip(p_mid * 0.90, config["auction"]["price_min"], config["auction"]["price_max"]))
        p2 = p_mid
        p3 = float(np.clip(p_mid * 1.10, config["auction"]["price_min"], config["auction"]["price_max"]))

        actions[i] = np.array(
            [p1, q_third, p2, q_third, p3, q_third, action6[2], action6[3], action6[4], action6[5]],
            dtype=np.float32,
        )

    return actions


def build_learning_secondary_actions(env: ETSEnvironment, config: dict) -> np.ndarray:
    actions = np.zeros((env.n_agents, 2), dtype=np.float32)

    for i in range(env.n_agents):
        company = env.companies[i]
        actions[i] = heuristic_policy.secondary_action(
            company,
            bank=float(env.holdings[i]),
            allocation=float(env._phase1_allocations[i]),
            clearing_price=env._phase1_clearing_price,
            config=config,
            current_year=env.current_year,
            n_years=config["simulation"]["n_years"],
        )

    return actions


def show_table(title: str, df: pd.DataFrame, digits: int = 3):
    print(title)
    print("-" * 120)
    if DISPLAY_YEAR_TABLES:
        display(df.round(digits))
    else:
        print(df.round(digits).to_string(index=False))


def print_preset_and_burnin_summary(config: dict, env: ETSEnvironment):
    if not PRINT_PRESET_SUMMARY:
        return

    warm = config.get("warm_start", {})
    msr_cfg = config["ets"]["msr"]
    tnac_mid = float(getattr(env.cap_schedule, "tnac_mid", env.cap_schedule.tnac_upper * (833.0 / 1096.0)))

    print("=" * 120)
    print("1. PRESET")
    print("=" * 120)
    print(f"Config path: {CONFIG_PATH}")
    print(f"Seed: {SEED}")
    print(f"Years: {config['simulation']['n_years']}")
    print(f"Participants: learning={env.n_agents}, bots={env.n_bots}, total={env.n_total}")
    print(
        f"Auction bounds: price=[{config['auction']['price_min']}, {config['auction']['price_max']}] | "
        f"qty_mult=[{config['auction'].get('qty_mult_low', 0.3)}, {config['auction'].get('qty_mult_high', 2.0)}]"
    )
    print(
        f"Penalty base={config['penalty']['rate']} | inflation_mean={config['penalty'].get('inflation_rate', 0.0)} | "
        f"reserve_mode={config['ets'].get('reserve_price_mode', 'static')} | reserve_floor={config['ets'].get('reserve_price', 0.0)}"
    )
    print(
        f"MSR: enabled={msr_cfg['enabled']} | tnac_upper={env.cap_schedule.tnac_upper:.3f} | "
        f"tnac_mid={tnac_mid:.3f} | tnac_lower={env.cap_schedule.tnac_lower:.3f} | "
        f"withhold_rate={env.cap_schedule.withhold_rate} | release={env.cap_schedule.release_amount:.3f}"
    )
    print("MSR intake regimes: above upper -> 24% of TNAC; middle band -> TNAC - tnac_mid; below lower -> fixed release.")
    print(
        f"Secondary market: tx_cost={config['trading'].get('transaction_cost', 0.0)} | "
        f"spread_tolerance={config['trading'].get('spread_tolerance', 0.0)}"
    )
    print(f"Initial expected price={config['price'].get('initial_expected', 0.0)}")

    print("")
    print("2. BURN-IN / START-UP")
    print("=" * 120)
    if warm.get("enabled", False) and warm.get("burnin_enabled", False):
        print(
            f"Hidden burn-in is enabled for {int(warm.get('n_burnin_years', 4))} years. "
            "The environment pre-runs synthetic years to seed prices, banks, queues, and MSR state."
        )
        print(f"Burn-in price seeds kept in history: {as_float_list(env._price_history, digits=3)}")
    elif warm.get("enabled", False):
        print(
            "Explicit burn-in is off, but warm start is enabled. The environment seeds starting banks, "
            "price history, and queues directly without hidden market years."
        )
        print(f"Seeded price history: {as_float_list(env._price_history, digits=3)}")
    else:
        print(
            "No explicit warm start or hidden burn-in is enabled in this config. The environment still seeds "
            "starting allowance banks near the MSR band so year 1 is not pathological."
        )
        print("Price history starts empty, and expected_price starts from the configured initial anchor.")

    print(f"Starting holdings (Mt): {as_float_list(env.holdings, digits=3)}")
    print("Starting participant balances and mixes:")
    for idx, company in enumerate(env.companies):
        print(
            f"  {participant_name(env, idx)} | bank={env.holdings[idx]:.3f} | annual_budget={company.annual_budget:.2f} | "
            f"capex_limit={company.capex_throughput:.2f} | green={company.green_frac * 100:.1f}% | mix: {fmt_mix(company)}"
        )


def explain_auction_supply(pre_tnac: float, log: dict, env: ETSEnvironment) -> list[str]:
    cap_t = float(log["cap"])
    auction_volume = float(log["auction_volume"])
    defaulted_rolled_in = float(log.get("defaulted_volume_rolled_in", 0.0))
    unsold_rollover_in = float(log.get("unsold_rollover_in", 0.0))
    msr_withheld = float(log.get("msr_withhold_this_year", 0.0))
    msr_released = float(log.get("msr_release_this_year", 0.0))
    msr_upper = float(env.cap_schedule.tnac_upper)
    msr_mid = float(getattr(env.cap_schedule, "tnac_mid", msr_upper * (833.0 / 1096.0)))
    msr_lower = float(env.cap_schedule.tnac_lower)

    reasons = [f"Start from the annual cap: {cap_t:.3f} Mt."]

    if pre_tnac > msr_upper + 1e-9:
        reasons.append(
            f"Starting TNAC {pre_tnac:.3f} is above MSR upper threshold {msr_upper:.3f}; baseline intake target is 24% of TNAC."
        )
    elif pre_tnac >= msr_mid - 1e-9:
        reasons.append(
            f"Starting TNAC {pre_tnac:.3f} is in the middle MSR band [{msr_mid:.3f}, {msr_upper:.3f}], where intake target is TNAC - tnac_mid."
        )
    elif pre_tnac < msr_lower - 1e-9:
        reasons.append(
            f"Starting TNAC {pre_tnac:.3f} is below MSR lower threshold {msr_lower:.3f}, so the MSR tends to release supply."
        )
    else:
        reasons.append(
            f"Starting TNAC {pre_tnac:.3f} sits in the neutral lower-to-mid band [{msr_lower:.3f}, {msr_mid:.3f}], so MSR pressure should be limited."
        )

    if msr_withheld > 1e-9:
        reasons.append(f"Applied MSR withholding this year: {msr_withheld:.3f} Mt.")
    else:
        reasons.append("Applied MSR withholding this year: 0.000 Mt.")

    if msr_released > 1e-9:
        reasons.append(f"Applied MSR release this year: {msr_released:.3f} Mt.")
    else:
        reasons.append("Applied MSR release this year: 0.000 Mt.")

    if unsold_rollover_in > 1e-9:
        reasons.append(f"Unsold allowances from last year added {unsold_rollover_in:.3f} Mt to this auction.")
    else:
        reasons.append("There was no meaningful unsold rollover coming into this auction.")

    if defaulted_rolled_in > 1e-9:
        reasons.append(f"Defaulted auction volume from last year added another {defaulted_rolled_in:.3f} Mt.")
    else:
        reasons.append("There was no defaulted volume carried into this auction.")

    base_plus_rollovers = cap_t + unsold_rollover_in + defaulted_rolled_in
    net_vs_base = auction_volume - base_plus_rollovers
    if net_vs_base < -1e-9:
        reasons.append(f"Net effect vs cap + rollovers is a withdrawal of {abs(net_vs_base):.3f} Mt.")
    elif net_vs_base > 1e-9:
        reasons.append(f"Net effect vs cap + rollovers is an addition of {net_vs_base:.3f} Mt.")
    else:
        reasons.append("Net effect vs cap + rollovers is essentially neutral.")

    reasons.append(f"Final offered auction volume this year is {auction_volume:.3f} Mt.")
    return reasons


def build_market_summary_df(pre_price: float, pre_expected_price: float, pre_tnac: float, pre_msr: float, log: dict, auction_stats: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "pre_price": pre_price,
                "expected_price": pre_expected_price,
                "pre_tnac": pre_tnac,
                "pre_msr": pre_msr,
                "cap_mt": log["cap"],
                "offered_mt": log["auction_volume"],
                "unsold_rollover_in": float(log.get("unsold_rollover_in", 0.0)),
                "defaulted_rollover_in": float(log.get("defaulted_volume_rolled_in", 0.0)),
                "msr_withhold_mt": float(log.get("msr_withhold_this_year", 0.0)),
                "msr_release_mt": float(log.get("msr_release_this_year", 0.0)),
                "reserve": log["effective_reserve"],
                "auction_clearing": log["clearing_price"],
                "total_demand": float(auction_stats.get("total_demand", 0.0)),
                "allocated_mt": float(auction_stats.get("total_allocated", 0.0)),
                "unsold_mt": float(auction_stats.get("unsold", 0.0)),
                "secondary_clearing": log["secondary_clearing"],
                "secondary_volume": log["secondary_volume"],
            }
        ]
    )


def build_financial_flow_df(env: ETSEnvironment, log: dict) -> pd.DataFrame:
    rows = []
    for i, company in enumerate(env.companies):
        annual_budget = float(company.annual_budget)
        budget_spent = float(company.budget_spent_this_year)
        budget_remaining = annual_budget - budget_spent
        capex_limit = float(company.capex_throughput)
        capex_spent = float(company.capex_spent_this_year)
        capex_remaining = capex_limit - capex_spent
        secondary_net_cash = -float(log["trade_costs"][i])
        tracked_spend = (
            float(log["payments"][i])
            + float(log["trade_costs"][i])
            + float(log["invest_costs"][i])
            + float(log.get("mac_costs", [0.0] * env.n_total)[i])
            + float(log.get("collateral_costs", [0.0] * env.n_total)[i])
            + float(log["penalties"][i])
        )
        rows.append(
            {
                "participant": participant_name(env, i),
                "annual_budget": annual_budget,
                "budget_spent": budget_spent,
                "budget_remaining": budget_remaining,
                "auction_payment": float(log["payments"][i]),
                "secondary_net_cash": secondary_net_cash,
                "investment_cost": float(log["invest_costs"][i]),
                "mac_cost": float(log.get("mac_costs", [0.0] * env.n_total)[i]),
                "collateral_cost": float(log.get("collateral_costs", [0.0] * env.n_total)[i]),
                "penalty_cost": float(log["penalties"][i]),
                "tracked_spend_check": tracked_spend,
                "capex_limit": capex_limit,
                "capex_spent": capex_spent,
                "capex_remaining": capex_remaining,
            }
        )
    return pd.DataFrame(rows)


def build_allowance_flow_df(env: ETSEnvironment, log: dict, carry_forward_start: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, company in enumerate(env.companies):
        start_bank = float(log["bank_start"][i])
        allocation = float(log["allocations"][i])
        secondary_trade = float(log["trade_qtys"][i])
        pre_compliance = start_bank + allocation + secondary_trade
        emissions = float(log["emissions"][i])
        carry_start = float(carry_forward_start[i])
        total_need = emissions + carry_start
        shortfall = float(log["shortfalls"][i])
        ending_bank = float(log["holdings"][i])
        next_carry_forward = float(company._carry_forward)
        coverage_ratio = pre_compliance / max(total_need, 1e-9)
        rows.append(
            {
                "participant": participant_name(env, i),
                "start_bank_mt": start_bank,
                "auction_alloc_mt": allocation,
                "secondary_trade_mt": secondary_trade,
                "pre_compliance_allowances_mt": pre_compliance,
                "realized_emissions_mt": emissions,
                "carry_forward_start_mt": carry_start,
                "total_need_end_mt": total_need,
                "coverage_ratio": coverage_ratio,
                "shortfall_mt": shortfall,
                "ending_bank_mt": ending_bank,
                "carry_forward_next_mt": next_carry_forward,
            }
        )
    return pd.DataFrame(rows)


def build_portfolio_df(env: ETSEnvironment, log: dict) -> pd.DataFrame:
    rows = []
    for i, company in enumerate(env.companies):
        tech_choice = int(log["invest_tech_choices"][i])
        tech_name = TECH_NAMES[tech_choice + 2] if 0 <= tech_choice <= 2 else str(tech_choice)
        rows.append(
            {
                "participant": participant_name(env, i),
                "green_frac_pct": 100.0 * float(log["green_fracs"][i]),
                "delta_green_pct": 100.0 * float(log.get("delta_greens", [0.0] * env.n_total)[i]),
                "invest_frac": float(log["invest_fracs"][i]),
                "invest_tech": tech_name,
                "queue_size": int(log.get("queue_sizes", [0] * env.n_total)[i]),
                "mix_summary": fmt_mix(company),
            }
        )
    return pd.DataFrame(rows)


def print_preset_tables(env: ETSEnvironment):
    rows = []
    for idx, company in enumerate(env.companies):
        rows.append(
            {
                "participant": participant_name(env, idx),
                "start_bank_mt": float(env.holdings[idx]),
                "annual_budget": float(company.annual_budget),
                "capex_limit": float(company.capex_throughput),
                "green_frac_pct": 100.0 * company.green_frac,
                "mix_summary": fmt_mix(company),
            }
        )
    show_table("Start-of-run participant overview", pd.DataFrame(rows), digits=3)


def print_auction_bids(log: dict, env: ETSEnvironment):
    print("5. AUCTION BIDS")
    print("-" * 120)
    for i in range(env.n_total):
        name = participant_name(env, i)
        tranches = [
            f"T{t + 1}(p={log['tranche_prices'][i][t]:.2f}, q={log['tranche_quantities'][i][t]:.4f})"
            for t in range(3)
        ]
        print(
            f"  {name:>3} | start_bank={log['bank_start'][i]:.3f} | est_need={log['estimate_needs'][i]:.3f} | "
            f"bid_qty={log['bid_quantities'][i]:.3f} | bid_wavg_price={log['bid_prices'][i]:.2f} | tranches={tranches}"
        )


def print_auction_winner_logic(log: dict, auction_stats: dict, env: ETSEnvironment):
    print("6. AUCTION CLEARING / WHO WINS")
    print("-" * 120)
    print("  Rule: sort valid bids by price descending, allocate until auction supply is exhausted, and all winners pay the same uniform clearing price.")
    print("  The clearing price is the lowest accepted bid. Ties at the margin are randomly ordered; only the last filled tied bid can be partial.")
    print(
        f"  clearing_price={log['clearing_price']:.2f} | total_demand={auction_stats.get('total_demand', 0.0):.3f} | "
        f"total_allocated={auction_stats.get('total_allocated', 0.0):.3f} | cover_ratio={auction_stats.get('cover_ratio', 0.0):.3f} | unsold={auction_stats.get('unsold', 0.0):.3f}"
    )
    if "hhi" in auction_stats:
        print(
            f"  concentration_hhi={auction_stats['hhi']:.2f} | max_agent_share_actual={auction_stats.get('max_agent_share_actual', 0.0):.3f}"
        )
    if auction_stats.get("auction_failed", False):
        print(f"  Auction failed: {auction_stats.get('fail_reason', 'unknown')}")
    if auction_stats.get("defaults", 0):
        print(
            f"  Post-clearing defaults={auction_stats.get('defaults', 0)} | defaulted_volume={auction_stats.get('defaulted_volume', 0.0):.3f} | "
            f"defaulted_agents={auction_stats.get('defaults_agents', [])}"
        )
    for i in range(env.n_total):
        if log["allocations"][i] > 1e-9:
            print(f"  WINNER {participant_name(env, i)} | allocation={log['allocations'][i]:.3f} | payment={log['payments'][i]:.3f}")


def print_shocks_and_state(log: dict, env: ETSEnvironment):
    print("7. SHOCKS / REALISATIONS")
    print("-" * 120)
    print("  Emission shocks and capacity-factor shocks are applied to generate realized emissions for the year.")
    print(f"  emission_shocks={as_float_list(log.get('emission_shocks', []), digits=5)}")
    print(f"  cf_shocks={as_float_list(log.get('cf_shocks', []), digits=5)}")
    print(f"  realized_emissions={as_float_list(log.get('emissions', []), digits=4)}")
    print(f"  mac_reductions={as_float_list(log.get('mac_reductions', []), digits=4)}")
    print(f"  mac_costs={as_float_list(log.get('mac_costs', []), digits=4)}")


def print_secondary_market(log: dict, env: ETSEnvironment):
    print("8. SECONDARY MARKET")
    print("-" * 120)
    print(
        f"  secondary_clearing={log['secondary_clearing']:.2f} | secondary_volume={log['secondary_volume']:.3f} | action_sides={log.get('sec_action_sides', [])}"
    )
    for i in range(env.n_total):
        print(
            f"  {participant_name(env, i):>3} | sec_price={log['sec_price_mults'][i]:.2f} | sec_order={log['sec_qty_actions'][i]:.3f} | "
            f"sec_fill={log['trade_qtys'][i]:.3f} | sec_cost={log['trade_costs'][i]:.3f}"
        )


def print_green_investments(log: dict, env: ETSEnvironment):
    print("9. GREEN INVESTMENT / PORTFOLIO CHANGE")
    print("-" * 120)
    print("  Note: the environment plans these in phase 1, but they are reported here after the market sections for chronological readability.")
    print(f"  cancellations={log.get('cancellations', [])}")
    print(f"  queue_sizes={log.get('queue_sizes', [])}")
    print(f"  delta_greens={as_float_list(log.get('delta_greens', []), digits=5)}")
    for i in range(env.n_total):
        tech_choice = int(log['invest_tech_choices'][i])
        tech_name = TECH_NAMES[tech_choice + 2] if 0 <= tech_choice <= 2 else str(tech_choice)
        print(
            f"  {participant_name(env, i):>3} | invest_frac={log['invest_fracs'][i]:.4f} | tech={tech_name} | invest_cost={log['invest_costs'][i]:.3f} | green_frac={100 * log['green_fracs'][i]:.2f}%"
        )


def print_compliance_and_wrap(log: dict, env: ETSEnvironment):
    print("10. COMPLIANCE / PENALTIES / REWARDS / WRAP-UP")
    print("-" * 120)
    print(f"  inflation_rate={log['inflation_rate']:.5f} | inflation_factor={log['inflation_factor']:.5f}")
    print(f"  collateral_costs={as_float_list(log.get('collateral_costs', []), digits=4)}")
    print(f"  penalties={as_float_list(log['penalties'], digits=4)}")
    print(f"  shortfalls={as_float_list(log['shortfalls'], digits=4)}")
    print(f"  rewards={as_float_list(log['rewards'], digits=4)}")
    print(f"  rewards_base={as_float_list(log.get('rewards_base', []), digits=4)}")
    print(f"  rewards_shaping={as_float_list(log.get('rewards_shaping', []), digits=4)}")
    print(f"  terminal_bank_values={as_float_list(log.get('terminal_bank_values', []), digits=4)}")
    print(f"  terminal_queue_values={as_float_list(log.get('terminal_queue_values', []), digits=4)}")
    print(f"  terminal_liquidation_values={as_float_list(log.get('terminal_liquidation_values', []), digits=4)}")
    print(f"  ending_holdings={as_float_list(log['holdings'], digits=4)}")
    for i in range(env.n_total):
        print(
            f"  {participant_name(env, i):>3} | end_bank={log['holdings'][i]:.3f} | shortfall={log['shortfalls'][i]:.3f} | penalty={log['penalties'][i]:.3f} | reward={log['rewards'][i]:.3f}"
        )


print_preset_and_burnin_summary(config, env)
print_preset_tables(env)

trace_rows = []

for year in range(config["simulation"]["n_years"]):
    pre_price = float(env.last_clearing_price)
    pre_expected_price = float(env.expected_price)
    pre_msr = float(env.cap_schedule.msr_reserve())
    pre_tnac = float(env.holdings.sum())
    # MSR decisions use a 1-year TNAC lag (cap_schedule._prev_tnac), not current pre_tnac.
    msr_decision_tnac = getattr(env.cap_schedule, "_prev_tnac", None)
    carry_forward_start = np.array([float(company._carry_forward) for company in env.companies], dtype=float)

    auction_actions = build_learning_auction_actions(env, config)
    obs_phase2, _ = env.step_auction(auction_actions)

    secondary_actions = build_learning_secondary_actions(env, config)
    obs_phase1, rewards, terminated, truncated, info = env.step_secondary(secondary_actions)
    log = info["year_log"]
    auction_stats = log.get("auction_stats", {})

    if PRINT_YEAR_HEADER:
        print("")
        print("#" * 120)
        print(f"YEAR {year + 1:02d} CHRONOLOGICAL TRACE")
        print("#" * 120)

    print("3. MARKET SETS UP / SUPPLY IS DECIDED")
    print("-" * 120)
    print(f"  pre_price={pre_price:.2f} | pre_expected_price={pre_expected_price:.2f} | pre_tnac={pre_tnac:.3f} | pre_msr={pre_msr:.3f}")
    if msr_decision_tnac is not None:
        print(
            f"  msr_decision_tnac_lagged={float(msr_decision_tnac):.3f} "
            "(MSR uses this lagged TNAC for intake/release decisions)"
        )
    for line in explain_auction_supply(
        float(pre_tnac if msr_decision_tnac is None else msr_decision_tnac),
        log,
        env,
    ):
        print(f"  {line}")
    if log.get("rollover_channels_equal", False):
        print(
            "  Note: unsold_rollover_in equals defaulted_volume_rolled_in this year; "
            "these channels can overlap in accounting."
        )
    show_table(
        "Market summary",
        build_market_summary_df(pre_price, pre_expected_price, pre_tnac, pre_msr, log, auction_stats),
        digits=3,
    )

    print("4. STARTING BALANCES")
    print("-" * 120)
    print(f"  starting_holdings={as_float_list(log['bank_start'], digits=4)}")
    for i in range(env.n_total):
        print(
            f"  {participant_name(env, i):>3} | start_bank={log['bank_start'][i]:.3f} | carry_in={carry_forward_start[i]:.3f} | mix: {fmt_mix(env.companies[i])}"
        )

    print_auction_bids(log, env)
    print_auction_winner_logic(log, auction_stats, env)
    print_shocks_and_state(log, env)
    print_secondary_market(log, env)
    print_green_investments(log, env)
    print_compliance_and_wrap(log, env)

    financial_flow_df = build_financial_flow_df(env, log)
    allowance_flow_df = build_allowance_flow_df(env, log, carry_forward_start)
    portfolio_df = build_portfolio_df(env, log)

    show_table("Financial flow by participant", financial_flow_df, digits=3)
    show_table("Allowance flow by participant", allowance_flow_df, digits=4)
    show_table("Portfolio and investment snapshot", portfolio_df, digits=3)

    if STORE_TRACE_TABLE:
        for i in range(env.n_total):
            tech_choice = int(log["invest_tech_choices"][i])
            tech_name = TECH_NAMES[tech_choice + 2] if 0 <= tech_choice <= 2 else str(tech_choice)
            trace_rows.append(
                {
                    "year": year + 1,
                    "participant": participant_name(env, i),
                    "pre_price": pre_price,
                    "pre_expected_price": pre_expected_price,
                    "pre_tnac": pre_tnac,
                    "msr_decision_tnac_lagged_mt": (
                        float("nan") if msr_decision_tnac is None else float(msr_decision_tnac)
                    ),
                    "pre_msr": pre_msr,
                    "starting_bank_mt": log["bank_start"][i],
                    "carry_forward_start_mt": carry_forward_start[i],
                    "annual_budget": float(env.companies[i].annual_budget),
                    "budget_spent": float(env.companies[i].budget_spent_this_year),
                    "budget_remaining": float(env.companies[i].annual_budget - env.companies[i].budget_spent_this_year),
                    "capex_limit": float(env.companies[i].capex_throughput),
                    "capex_spent": float(env.companies[i].capex_spent_this_year),
                    "capex_remaining": float(env.companies[i].capex_throughput - env.companies[i].capex_spent_this_year),
                    "cap_mt": log["cap"],
                    "auction_volume_mt": log["auction_volume"],
                    "unsold_rollover_in_mt": float(log.get("unsold_rollover_in", 0.0)),
                    "defaulted_rollover_in_mt": float(log.get("defaulted_volume_rolled_in", 0.0)),
                    "msr_withhold_mt": float(log.get("msr_withhold_this_year", 0.0)),
                    "msr_release_mt": float(log.get("msr_release_this_year", 0.0)),
                    "effective_reserve": log["effective_reserve"],
                    "auction_clearing": log["clearing_price"],
                    "auction_failed": bool(auction_stats.get("auction_failed", False)),
                    "total_demand": float(auction_stats.get("total_demand", 0.0)),
                    "total_allocated": float(auction_stats.get("total_allocated", 0.0)),
                    "cover_ratio": float(auction_stats.get("cover_ratio", 0.0)),
                    "unsold": float(auction_stats.get("unsold", 0.0)),
                    "hhi": float(auction_stats.get("hhi", 0.0)),
                    "auction_defaults": int(auction_stats.get("defaults", 0)),
                    "auction_defaulted_volume": float(auction_stats.get("defaulted_volume", 0.0)),
                    "estimate_need_mt": log["estimate_needs"][i],
                    "bid_qty_multiplier": log["bid_qty_multipliers"][i],
                    "bid_quantity_mt": log["bid_quantities"][i],
                    "bid_weighted_price": log["bid_prices"][i],
                    "tranche_prices": log["tranche_prices"][i],
                    "tranche_quantities": log["tranche_quantities"][i],
                    "allocation_mt": log["allocations"][i],
                    "payment_meur": log["payments"][i],
                    "secondary_clearing": log["secondary_clearing"],
                    "secondary_volume": log["secondary_volume"],
                    "secondary_action_price": log["sec_price_mults"][i],
                    "secondary_action_qty": log["sec_qty_actions"][i],
                    "secondary_trade_qty": log["trade_qtys"][i],
                    "secondary_trade_cost": log["trade_costs"][i],
                    "secondary_net_cash": -float(log["trade_costs"][i]),
                    "investment_cost": log["invest_costs"][i],
                    "invest_frac": log["invest_fracs"][i],
                    "invest_tech_choice": tech_name,
                    "green_fraction": log["green_fracs"][i],
                    "queue_size": log.get("queue_sizes", [0] * env.n_total)[i],
                    "delta_green": log.get("delta_greens", [0.0] * env.n_total)[i],
                    "emission_shock": log.get("emission_shocks", [0.0] * env.n_total)[i],
                    "cf_shock": log.get("cf_shocks", [0.0] * env.n_total)[i],
                    "emissions_mt": log["emissions"][i],
                    "mac_reduction": log.get("mac_reductions", [0.0] * env.n_total)[i],
                    "mac_cost": log.get("mac_costs", [0.0] * env.n_total)[i],
                    "collateral_cost": log.get("collateral_costs", [0.0] * env.n_total)[i],
                    "pre_compliance_allowances_mt": float(log["bank_start"][i] + log["allocations"][i] + log["trade_qtys"][i]),
                    "total_need_end_mt": float(log["emissions"][i] + carry_forward_start[i]),
                    "shortfall_mt": log["shortfalls"][i],
                    "carry_forward_next_mt": float(env.companies[i]._carry_forward),
                    "penalty_meur": log["penalties"][i],
                    "reward": log["rewards"][i],
                    "reward_base": log.get("rewards_base", [0.0] * env.n_total)[i],
                    "reward_shaping": log.get("rewards_shaping", [0.0] * env.n_total)[i],
                    "terminal_bank_value": log.get("terminal_bank_values", [0.0] * env.n_total)[i],
                    "terminal_queue_value": log.get("terminal_queue_values", [0.0] * env.n_total)[i],
                    "terminal_liquidation_value": log.get("terminal_liquidation_values", [0.0] * env.n_total)[i],
                    "ending_bank_mt": log["holdings"][i],
                    "inflation_rate": log["inflation_rate"],
                    "inflation_factor": log["inflation_factor"],
                }
            )

    if terminated or truncated:
        break

if PRINT_FINAL_SUMMARY:
    print("")
    print("=" * 120)
    print("FINAL EPISODE SUMMARY")
    print("=" * 120)
    for i, company in enumerate(env.companies):
        print(
            f"  {participant_name(env, i):>3} | final_bank={env.holdings[i]:.3f} | final_carry_forward={company._carry_forward:.3f} | "
            f"budget_spent={company.budget_spent_this_year:.3f} | green={company.green_frac * 100:.1f}% | mix: {fmt_mix(company)}"
        )

if PRINT_WARNING_COUNTERS and hasattr(env, "_warnings"):
    print("")
    print("=" * 120)
    print("WARNING COUNTERS")
    print("=" * 120)
    for k in sorted(env._warnings.keys()):
        print(f"  {k}: {env._warnings[k]}")

1. PRESET
Config path: c:\Users\danie\Documents\Python_Scripts\Master Thesis\Thesis-Energy-Auction\ets_marl_happo_auction\configs\default.yaml
Seed: 42
Years: 12
Participants: learning=1, bots=15, total=16
Auction bounds: price=[30.0, 500.0] | qty_mult=[0.1, 1.5]
Penalty base=138.75 | inflation_mean=0.02 | reserve_mode=static | reserve_floor=30.0
MSR: enabled=True | tnac_upper=18.215 | tnac_mid=13.844 | tnac_lower=6.648 | withhold_rate=0.24 | release=3.230
MSR intake regimes: above upper -> 24% of TNAC; middle band -> TNAC - tnac_mid; below lower -> fixed release.
Secondary market: tx_cost=0.5 | spread_tolerance=0.12
Initial expected price=80.0

2. BURN-IN / START-UP
Hidden burn-in is enabled for 4 years. The environment pre-runs synthetic years to seed prices, banks, queues, and MSR state.
Burn-in price seeds kept in history: [121.706, 121.706, 122.216]
Starting holdings (Mt): [1.385, 1.264, 1.421, 1.033, 0.928, 0.389, 0.454, 0.201, 0.155, 1.039, 1.528, 0.807, 1.01, 0.339, 0.288, 0.19

,participant,start_bank_mt,annual_budget,capex_limit,green_frac_pct,mix_summary
0,A1,1.385,1010.0,150.0,20.000,"coal=40.0%, gas=40.0%, onshore_wind=10.0%, off..."
1,B1,1.264,1010.0,150.0,20.000,"coal=40.0%, gas=40.0%, onshore_wind=10.0%, off..."
2,B2,1.421,1010.0,150.0,23.992,"coal=36.0%, gas=40.0%, onshore_wind=10.0%, off..."
3,B3,1.033,920.0,150.0,40.736,"coal=14.3%, gas=45.0%, onshore_wind=20.0%, off..."
4,B4,0.928,920.0,150.0,40.000,"coal=15.0%, gas=45.0%, onshore_wind=20.0%, off..."
5,B5,0.389,945.0,185.0,75.589,"coal=0.0%, gas=24.4%, onshore_wind=37.5%, offs..."
6,B6,0.454,945.0,185.0,70.707,"coal=4.3%, gas=25.0%, onshore_wind=35.0%, offs..."
7,B7,0.201,900.0,140.0,90.000,"coal=0.0%, gas=10.0%, onshore_wind=30.0%, offs..."
8,B8,0.155,900.0,140.0,90.000,"coal=0.0%, gas=10.0%, onshore_wind=30.0%, offs..."
9,B9,1.039,1010.0,150.0,20.000,"coal=40.0%, gas=40.0%, onshore_wind=10.0%, off..."



########################################################################################################################
YEAR 01 CHRONOLOGICAL TRACE
########################################################################################################################
3. MARKET SETS UP / SUPPLY IS DECIDED
------------------------------------------------------------------------------------------------------------------------
  pre_price=122.22 | pre_expected_price=121.48 | pre_tnac=12.432 | pre_msr=4.030
  msr_decision_tnac_lagged=17.874 (MSR uses this lagged TNAC for intake/release decisions)
  Start from the annual cap: 50.598 Mt.
  Starting TNAC 17.874 is in the middle MSR band [13.844, 18.215], where intake target is TNAC - tnac_mid.
  Applied MSR withholding this year: 4.030 Mt.
  Applied MSR release this year: 0.000 Mt.
  There was no meaningful unsold rollover coming into this auction.
  There was no defaulted volume carried into this auction.
  Net effect vs cap + rollovers is

,pre_price,expected_price,pre_tnac,pre_msr,cap_mt,offered_mt,unsold_rollover_in,defaulted_rollover_in,msr_withhold_mt,msr_release_mt,reserve,auction_clearing,total_demand,allocated_mt,unsold_mt,secondary_clearing,secondary_volume
0,122.216,121.477,12.432,4.03,50.598,46.568,0.0,0.0,4.03,0.0,30.0,101.234,51.599,46.568,0.0,101.234,0.0


4. STARTING BALANCES
------------------------------------------------------------------------------------------------------------------------
  starting_holdings=[1.385, 1.2641, 1.4214, 1.0326, 0.9282, 0.3886, 0.454, 0.2014, 0.1553, 1.0392, 1.5278, 0.8075, 1.0097, 0.3388, 0.2875, 0.1906]
   A1 | start_bank=1.385 | carry_in=0.000 | mix: coal=40.0%, gas=40.0%, onshore_wind=10.0%, offshore_wind=5.0%, solar=5.0%
   B1 | start_bank=1.264 | carry_in=0.000 | mix: coal=40.0%, gas=40.0%, onshore_wind=10.0%, offshore_wind=5.0%, solar=5.0%
   B2 | start_bank=1.421 | carry_in=0.000 | mix: coal=36.0%, gas=40.0%, onshore_wind=10.0%, offshore_wind=5.0%, solar=9.0%
   B3 | start_bank=1.033 | carry_in=0.000 | mix: coal=14.3%, gas=45.0%, onshore_wind=20.0%, offshore_wind=10.0%, solar=10.7%
   B4 | start_bank=0.928 | carry_in=0.000 | mix: coal=15.0%, gas=45.0%, onshore_wind=20.0%, offshore_wind=10.0%, solar=10.0%
   B5 | start_bank=0.389 | carry_in=0.000 | mix: coal=0.0%, gas=24.4%, onshore_wind=37.5%, o

KeyError: 'tranche_prices'

In [ ]:
if STORE_TRACE_TABLE and trace_rows:
    trace_df = pd.DataFrame(trace_rows)
    print(f"Trace rows: {len(trace_df)}")
    display(trace_df.head(10))

    # Optional export
    # out_path = PROJECT_ROOT / "results" / f"debug_trace_seed{SEED}.csv"
    # trace_df.to_csv(out_path, index=False)
    # print(f"Wrote trace to {out_path}")

Trace rows: 192


,year,participant,pre_price,pre_expected_price,pre_tnac,msr_decision_tnac_lagged_mt,pre_msr,starting_bank_mt,carry_forward_start_mt,annual_budget,...,penalty_meur,reward,reward_base,reward_shaping,terminal_bank_value,terminal_queue_value,terminal_liquidation_value,ending_bank_mt,inflation_rate,inflation_factor
0,1,A1,121.73748,121.040404,13.528534,38.707312,25.568005,0.105620,0.0,880.0,...,132.391418,-1.343720,-1.343720,0.000000,0.0,0.0,0.0,0.000000,0.024571,1.0
1,1,B1,121.73748,121.040404,13.528534,38.707312,25.568005,1.832271,0.0,880.0,...,0.000000,-1.235271,-1.235271,0.000000,0.0,0.0,0.0,0.885759,0.024571,1.0
2,1,B2,121.73748,121.040404,13.528534,38.707312,25.568005,1.828256,0.0,880.0,...,0.000000,-0.440326,-0.549642,0.109316,0.0,0.0,0.0,1.166034,0.024571,1.0
3,1,B3,121.73748,121.040404,13.528534,38.707312,25.568005,1.219494,0.0,800.0,...,0.000000,-1.012249,-1.042510,0.030262,0.0,0.0,0.0,1.616704,0.024571,1.0
4,1,B4,121.73748,121.040404,13.528534,38.707312,25.568005,1.220319,0.0,800.0,...,0.000000,-0.620835,-0.620835,0.000000,0.0,0.0,0.0,2.924784,0.024571,1.0
5,1,B5,121.73748,121.040404,13.528534,38.707312,25.568005,0.346366,0.0,820.0,...,0.000000,-0.194355,-0.652058,0.457703,0.0,0.0,0.0,0.329323,0.024571,1.0
6,1,B6,121.73748,121.040404,13.528534,38.707312,25.568005,0.613217,0.0,820.0,...,0.000000,-0.327322,-0.385083,0.057761,0.0,0.0,0.0,1.490612,0.024571,1.0
7,1,B7,121.73748,121.040404,13.528534,38.707312,25.568005,0.148556,0.0,780.0,...,0.000000,-0.500081,-0.500081,0.000000,0.0,0.0,0.0,0.083383,0.024571,1.0
8,1,B8,121.73748,121.040404,13.528534,38.707312,25.568005,0.237695,0.0,780.0,...,0.000000,-0.253441,-0.253441,0.000000,0.0,0.0,0.0,0.390581,0.024571,1.0
9,1,B9,121.73748,121.040404,13.528534,38.707312,25.568005,1.395230,0.0,880.0,...,0.000000,-1.247999,-1.247999,0.000000,0.0,0.0,0.0,0.647797,0.024571,1.0
